# Ollama em GPU para a amostragem on-policy (passo 14)

Na CPU local cada amostra do aluno leva cerca de 50 s; um job de 400 prompts × 4 amostras levaria perto de um dia e não tem retomada. Este notebook serve o **mesmo GGUF do aluno** num Ollama com a T4 do Colab e o expõe por um túnel temporário, para a API local usar como `LOCAL_LLM_BASE_URL` só durante o job on-policy.

**O que continua local:** a API, o juiz, os arquivos do job. **O que vem para cá:** só a geração das amostras.

**Cuidados:**

- O túnel dá uma URL pública **sem autenticação**. Não a divulgue e rode a última célula ao terminar.
- Por ele passam o prompt do aluno e as perguntas do split de treino (dados sintéticos, sem dado pessoal).
- A **avaliação** (passo 11) continua na CPU local: é o cenário on-premise que a pesquisa mede. Volte o `LOCAL_LLM_BASE_URL` para `http://localhost:11434/v1` e reinicie a API antes de avaliar.

**Como usar:** `Ambiente de execução → Alterar tipo → GPU (T4)`, ajuste `MODELO` se preciso e execute as células em ordem.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
assert subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0, "Sessao sem GPU: selecione a T4."


## 1. Instalar e subir o Ollama

`OLLAMA_NUM_PARALLEL` é quantas requisições o modelo atende ao mesmo tempo; use o mesmo valor em `concorrencia` no corpo do job.

In [ ]:
import os, time, urllib.request
PARALELO = 4
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
ambiente = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_NUM_PARALLEL=str(PARALELO), OLLAMA_KEEP_ALIVE="24h")
servidor = subprocess.Popen(["ollama", "serve"], env=ambiente, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("o Ollama nao subiu; veja ollama.log")
print("Ollama no ar")


## 2. Enviar o modelo do aluno

Envie `modelo-q4_k_m.gguf` e `Modelfile` da pasta da condição **C3** (os mesmos usados no `ollama create` local). O envio de ~1 GB pelo navegador leva alguns minutos; se os arquivos estiverem no Google Drive, preencha `PASTA_NO_DRIVE`.

`MODELO` tem de ser o nome que o aluno tem no Ollama local (o `modelo_local_id` sem `local:`): é o id que o job grava.

In [ ]:
MODELO = "qwen3-1.7b-lgpd-5b1301bb"
PASTA_NO_DRIVE = "/content/drive/MyDrive/modelo_tcc_ufg"  # ex.: "/content/drive/MyDrive/TCC/Filtrado_QWEN"; vazio = enviar pelo navegador
import os, shutil
os.makedirs("/content/modelo", exist_ok=True)
%cd /content/modelo
ARQUIVOS = ("modelo-q4_k_m.gguf", "Modelfile")
faltando = [nome for nome in ARQUIVOS if not os.path.exists(nome)]
if not faltando:
    print("arquivos ja' estao em /content/modelo; nada a enviar")
elif PASTA_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for nome in faltando:
        shutil.copy(os.path.join(PASTA_NO_DRIVE, nome), nome)
else:
    from google.colab import files
    print("envie:", ", ".join(faltando))
    files.upload()
assert os.path.exists("modelo-q4_k_m.gguf") and os.path.exists("Modelfile"), "faltou modelo-q4_k_m.gguf ou Modelfile"
!ollama create $MODELO -f Modelfile


## 3. Conferir que o modelo responde e que está na GPU

A resposta deve vir em português citando ids `LGPD:art.N`, e `ollama ps` deve mostrar `100% GPU`. Se mostrar CPU, o job seria tão lento quanto o local: não prossiga.

In [ ]:
import json
corpo = json.dumps({"model": MODELO, "messages": [{"role": "user", "content": "O que e dado pessoal sensivel?"}], "temperature": 0}).encode()
inicio = time.time()
req = urllib.request.Request("http://127.0.0.1:11434/v1/chat/completions", data=corpo, headers={"Content-Type": "application/json"})
resposta = json.loads(urllib.request.urlopen(req, timeout=300).read())
print(resposta["choices"][0]["message"]["content"])
print(f"\n{resposta['usage']['completion_tokens']} tokens em {time.time() - inicio:.0f}s")
!ollama ps


## 4. Abrir o túnel

A célula imprime a linha a colocar no `.env` da API local. Depois: reinicie a API (`uvicorn main:app`, sem `--reload`), confira em `POST /api/v1/treinos/{id}/verificar` que o aluno aparece como disponível e dispare o job on-policy com `"concorrencia": 4`. Mantenha esta aba aberta até o job terminar.

In [ ]:
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
# --http-host-header: o Ollama recusa (403) requisicoes cujo Host nao seja local.
tunel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "127.0.0.1:11434"],
                         stdout=open("/content/tunel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    achado = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/tunel.log").read())
    if achado:
        URL = achado.group(0)
        break
    time.sleep(1)
assert URL, "o tunel nao abriu; veja /content/tunel.log"
print("No .env da API local (e reinicie a API):\n")
print(f"LOCAL_LLM_BASE_URL={URL}/v1")


## 5. Manter a sessão viva e acompanhar

Deixe rodando durante o job: imprime a cada 5 minutos o estado do modelo. Interrompa a célula quando o job on-policy concluir.

In [ ]:
while True:
    print(time.strftime("%H:%M:%S"), subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout.strip().replace("\n", " | "))
    time.sleep(300)


## 6. Fechar o túnel

Rode ao terminar. Depois, no `.env` local, volte `LOCAL_LLM_BASE_URL=http://localhost:11434/v1` e reinicie a API.

In [ ]:
tunel.terminate()
servidor.terminate()
print("tunel e Ollama encerrados")
